# Classifying User Gender Based on Tweet Text
By: ED KING, [Post Link](https://www.kaggle.com/code/kinguistics/classifying-user-gender-based-on-tweet-text/notebook?select=gender-classifier-DFE-791531.csv)

## Download Dataset from Kaggle

In [16]:
# source: https://www.kaggle.com/discussions/general/74235
from google.colab import userdata
import os

os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

!kaggle datasets download -d crowdflower/twitter-user-gender-classification

! unzip "twitter-user-gender-classification.zip"

Dataset URL: https://www.kaggle.com/datasets/crowdflower/twitter-user-gender-classification
License(s): CC0-1.0
twitter-user-gender-classification.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  twitter-user-gender-classification.zip
replace gender-classifier-DFE-791531.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: gender-classifier-DFE-791531.csv  


In [40]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt # we'll want this for plotting
import re # we'll want this for text manipulation

from collections import defaultdict # for quick counting

# the Naive Bayes model
from sklearn.naive_bayes import MultinomialNB                # Classifier
from sklearn.model_selection import train_test_split

# function for transforming documents into counts
from sklearn.feature_extraction.text import CountVectorizer  # Vectorizer
from sklearn.preprocessing import LabelEncoder # function for encoding categories

from sklearn import metrics

# TASK: investigate why latin1 must be used?
# have to use latin1 even though it results in a lot of dead characters
twigen = pd.read_csv("gender-classifier-DFE-791531.csv", encoding='latin1')
print(len(twigen))
twigen.head()

20050


,_unit_id,_golden,_unit_state,_trusted_judgments,_last_judgment_at,gender,gender:confidence,profile_yn,profile_yn:confidence,created,...,profileimage,retweet_count,sidebar_color,text,tweet_coord,tweet_count,tweet_created,tweet_id,tweet_location,user_timezone
0,815719226,False,finalized,3,10/26/15 23:24,male,1.0000,yes,1.0,12/5/13 1:48,...,https://pbs.twimg.com/profile_images/414342229...,0,FFFFFF,Robbie E Responds To Critics After Win Against...,NaN,110964,10/26/15 12:40,6.587300e+17,main; @Kan1shk3,Chennai
1,815719227,False,finalized,3,10/26/15 23:30,male,1.0000,yes,1.0,10/1/12 13:51,...,https://pbs.twimg.com/profile_images/539604221...,0,C0DEED,ÛÏIt felt like they were my friends and I was...,NaN,7471,10/26/15 12:40,6.587300e+17,NaN,Eastern Time (US & Canada)
2,815719228,False,finalized,3,10/26/15 23:33,male,0.6625,yes,1.0,11/28/14 11:30,...,https://pbs.twimg.com/profile_images/657330418...,1,C0DEED,i absolutely adore when louis starts the songs...,NaN,5617,10/26/15 12:40,6.587300e+17,clcncl,Belgrade
3,815719229,False,finalized,3,10/26/15 23:10,male,1.0000,yes,1.0,6/11/09 22:39,...,https://pbs.twimg.com/profile_images/259703936...,0,C0DEED,Hi @JordanSpieth - Looking at the url - do you...,NaN,1693,10/26/15 12:40,6.587300e+17,"Palo Alto, CA",Pacific Time (US & Canada)
4,815719230,False,finalized,3,10/27/15 1:15,female,1.0000,yes,1.0,4/16/14 13:23,...,https://pbs.twimg.com/profile_images/564094871...,0,0,Watching Neighbours on Sky+ catching up with t...,NaN,31462,10/26/15 12:40,6.587300e+17,NaN,NaN


# Preprocessing

In [49]:
def normalize_text(s):
    # just in case
    s = str(s)
    s = s.lower()

    # remove punctuation that is not word-internal (e.g., hyphens, apostrophes)
    s = re.sub('\\s\\W',' ',s)
    s = re.sub('\\W\\s',' ',s)

    # make sure we didn't introduce any double spaces
    s = re.sub('\\s+',' ',s)

    return s

twigen['text_norm'] = [normalize_text(s) for s in twigen['text']]
twigen['description_norm'] = [normalize_text(s) for s in twigen['description']]

In [30]:
twigen.head()

,_unit_id,_golden,_unit_state,_trusted_judgments,_last_judgment_at,gender,gender:confidence,profile_yn,profile_yn:confidence,created,...,sidebar_color,text,tweet_coord,tweet_count,tweet_created,tweet_id,tweet_location,user_timezone,text_norm,description_norm
0,815719226,False,finalized,3,10/26/15 23:24,male,1.0000,yes,1.0,12/5/13 1:48,...,FFFFFF,Robbie E Responds To Critics After Win Against...,NaN,110964,10/26/15 12:40,6.587300e+17,main; @Kan1shk3,Chennai,robbie e responds to critics after win against...,i sing my own rhythm.
1,815719227,False,finalized,3,10/26/15 23:30,male,1.0000,yes,1.0,10/1/12 13:51,...,C0DEED,ÛÏIt felt like they were my friends and I was...,NaN,7471,10/26/15 12:40,6.587300e+17,NaN,Eastern Time (US & Canada),ûïit felt like they were my friends and i was...,i'm the author of novels filled with family dr...
2,815719228,False,finalized,3,10/26/15 23:33,male,0.6625,yes,1.0,11/28/14 11:30,...,C0DEED,i absolutely adore when louis starts the songs...,NaN,5617,10/26/15 12:40,6.587300e+17,clcncl,Belgrade,i absolutely adore when louis starts the songs...,louis whining and squealing and all
3,815719229,False,finalized,3,10/26/15 23:10,male,1.0000,yes,1.0,6/11/09 22:39,...,C0DEED,Hi @JordanSpieth - Looking at the url - do you...,NaN,1693,10/26/15 12:40,6.587300e+17,"Palo Alto, CA",Pacific Time (US & Canada),hi jordanspieth looking at the url do you use ...,mobile guy 49ers shazam google kleiner perkins...
4,815719230,False,finalized,3,10/27/15 1:15,female,1.0000,yes,1.0,4/16/14 13:23,...,0,Watching Neighbours on Sky+ catching up with t...,NaN,31462,10/26/15 12:40,6.587300e+17,NaN,NaN,watching neighbours on sky catching up with th...,ricky wilson the best frontman/kaiser chiefs t...


Let's grab some info about the gold standard and about the dataset's confidence in its gender classifications so we have some idea of what would be good to train on.

In [50]:

# how many observations are gold standard?
# golden values(True,False) mean that the entry is 100% reliable and verified
# Source: https://www.geeksforgeeks.org/python/defaultdict-in-python/
gold_values = defaultdict(int)
for val in twigen._golden:
    gold_values[val] += 1
print(gold_values)

print(np.any(np.isnan(twigen['gender:confidence']))) # what does the confidence look like?

gender_confidence = twigen['gender:confidence'][
    np.where(np.invert(np.isnan(twigen['gender:confidence'])))[0]] # we've got at least one NaN, so let's remove

print(len(gender_confidence))

# IMPORTANT: Debug the lines below
# gender_nonones = gender_confidence[np.where(gender_confidence < 1)[0]]
# print(len(gender_nonones))

defaultdict(<class 'int'>, {False: 20000, True: 50})
True
20024


About 30% of the observations have less than 100% confidence in the gender classification, so we'll ignore those.

In [51]:
twigen_confident = twigen[twigen['gender:confidence']==1]
twigen_confident.shape

twigen_confident.head()

,_unit_id,_golden,_unit_state,_trusted_judgments,_last_judgment_at,gender,gender:confidence,profile_yn,profile_yn:confidence,created,...,sidebar_color,text,tweet_coord,tweet_count,tweet_created,tweet_id,tweet_location,user_timezone,text_norm,description_norm
0,815719226,False,finalized,3,10/26/15 23:24,male,1.0,yes,1.0,12/5/13 1:48,...,FFFFFF,Robbie E Responds To Critics After Win Against...,NaN,110964,10/26/15 12:40,6.587300e+17,main; @Kan1shk3,Chennai,robbie e responds to critics after win against...,i sing my own rhythm.
1,815719227,False,finalized,3,10/26/15 23:30,male,1.0,yes,1.0,10/1/12 13:51,...,C0DEED,ÛÏIt felt like they were my friends and I was...,NaN,7471,10/26/15 12:40,6.587300e+17,NaN,Eastern Time (US & Canada),ûïit felt like they were my friends and i was...,i'm the author of novels filled with family dr...
3,815719229,False,finalized,3,10/26/15 23:10,male,1.0,yes,1.0,6/11/09 22:39,...,C0DEED,Hi @JordanSpieth - Looking at the url - do you...,NaN,1693,10/26/15 12:40,6.587300e+17,"Palo Alto, CA",Pacific Time (US & Canada),hi jordanspieth looking at the url do you use ...,mobile guy 49ers shazam google kleiner perkins...
4,815719230,False,finalized,3,10/27/15 1:15,female,1.0,yes,1.0,4/16/14 13:23,...,0,Watching Neighbours on Sky+ catching up with t...,NaN,31462,10/26/15 12:40,6.587300e+17,NaN,NaN,watching neighbours on sky catching up with th...,ricky wilson the best frontman/kaiser chiefs t...
5,815719231,False,finalized,3,10/27/15 1:47,female,1.0,yes,1.0,3/11/10 18:14,...,0,"Ive seen people on the train with lamps, chair...",NaN,20036,10/26/15 12:40,6.587300e+17,New York Gritty,Central Time (US & Canada),ive seen people on the train with lamps chairs...,you don't know me.


Okay, now let's see how well a Naive Bayes classifier can do by just looking at the words in the randomly chosen tweet.

In [35]:
# pull the data into vectors
vectorizer = CountVectorizer()
x = vectorizer.fit_transform(twigen_confident['text_norm'])

encoder = LabelEncoder()
y = encoder.fit_transform(twigen_confident['gender'])

# split into train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

# take a look at the shape of each of these
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)


(11140, 33868)
(11140,)
(2786, 33868)
(2786,)


# Training

In [36]:
nb = MultinomialNB()
nb.fit(x_train, y_train)

print(nb.score(x_test, y_test))

0.5778894472361809


So we get about 58% accuracy on the "best" observations, using only tweet text.

Let's try a couple more features. Specifically, let's add the description text by concatenating it to the tweet text.

In [53]:
twigen['all_features'] = twigen['text_norm'].str.cat(twigen['description_norm'], sep=' ')

twigen_confident = twigen[twigen['gender:confidence']==1]

In [56]:
# pull the data into vectors
vectorizer = CountVectorizer()
x = vectorizer.fit_transform(twigen_confident['all_features'])

encoder = LabelEncoder()
y = encoder.fit_transform(twigen_confident['gender'])

# split into train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

nb = MultinomialNB()
nb.fit(x_train, y_train)

print(nb.score(x_test, y_test))

0.6744436468054559


Cool, so we gain about 2-3 percentage points in accuracy just by adding description text alongside tweet text.

You can use this kind of procedure to play around with adding more features, or try a different type of model and see how accurately you can predict gender. (Maybe also try including the less-confident observations; my exclusion of them was probably anti-conservative)

# Visualizations and Reports

In [45]:
# DEBUG: What are the extra 2 classes brand and unknown? Do they come from the Dataset?
predicted = nb.predict(x_test)

print(
    f"Classification report for Kaggle dataset:\n"
    f"{metrics.classification_report(y_test, predicted, target_names=encoder.classes_)}\n"
)

Classification report for Kaggle dataset:
              precision    recall  f1-score   support

       brand       0.68      0.65      0.67       743
      female       0.57      0.73      0.64      1095
        male       0.50      0.37      0.42       926
     unknown       0.00      0.00      0.00        22

    accuracy                           0.58      2786
   macro avg       0.44      0.44      0.43      2786
weighted avg       0.57      0.58      0.57      2786




/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
